# 10 — Synthetic Data Generation

More data, better model — but only if the data is correct. This notebook generates
synthetic ARO programs at scale and uses the actual runtime as the quality signal.

The model explores many candidates at different temperatures (GRPO-style), scores
them with `aro check` + `aro run`, and saves only the ones that work. Every 30 valid
samples, a LoRA fine-tune round runs on all accumulated successes and the improved
adapter is hot-reloaded — so the model gets better at ARO *while it generates*.

**Why the warm-start matters here:** a cold base model fails `aro check` on most
attempts, so the explore loop never accumulates enough data to fine-tune.
The warm-start adapter from notebook 04 gives the model enough ARO fluency to
succeed often enough for the RL loop to kick in and compound.

**Inputs:**
- `../data/02_knowledge/knowledge.json`
- `../data/02_knowledge/knowledge_pairs.jsonl` — curated pairs used as few-shot context

**Output:** `../data/03_raw_generated/samples.jsonl`

**Seven task types (targets vs actual last run):**

| Task | Target | Actual |
|------|-------:|-------:|
| `code_generation` | 200 | 361 |
| `syntax_qa` | 150 | 150 |
| `fim` | 200 | 200 |
| `code_explanation` | 150 | 104 |
| `debugging` | 120 | 62 |
| `translation` | 80 | 2 |
| `architecture` | 60 | 1 |
| **Total** | **960** | **880** |

Under-target tasks (translation, architecture, debugging) reflect lower pass rates
through `aro check` — they require more valid ARO structure and the model struggles
more at these. Over-target code_generation reflects higher success rates.

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'mlx-lm'], check=False)

In [ ]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')
print(f'ARO root:  {ARO_ROOT}')
print(f'Pairs:     {PAIRS_FILE}')
print(f'Adapters:  {ADAPTER_DIR}')

import json
import re
import sys
import random
import subprocess
import tempfile
import importlib
from pathlib import Path
from collections import Counter

def ensure_mlx_lm():
    try:
        from mlx_lm import load, generate as mlx_generate
        return load, mlx_generate
    except ModuleNotFoundError:
        print("mlx_lm not found in the current kernel environment. Installing...")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "mlx-lm"],
            capture_output=True,
            text=True
        )
        if result.returncode != 0:
            raise RuntimeError(
                "Failed to install mlx-lm in the current Python environment.\n"
                f"stdout:\n{result.stdout}\n\nstderr:\n{result.stderr}"
            )

        importlib.invalidate_caches()

        try:
            from mlx_lm import load, generate as mlx_generate
            return load, mlx_generate
        except ModuleNotFoundError as e:
            raise RuntimeError(
                "mlx-lm was installed, but the current Jupyter kernel still cannot import it.\n"
                "Please restart the notebook kernel and rerun the notebook."
            ) from e

load, mlx_generate = ensure_mlx_lm()

DATA_OUT = DATA_ROOT / '03_raw_generated'
DATA_OUT.mkdir(parents=True, exist_ok=True)

# --- Model config ---
# MoE model: 30B params but only ~3.3B active — fits in 8 GB unified memory.

# Target: 500+ samples per run.
# Distribution: 60% code_generation, 30% debugging, rest spread across other tasks.
TARGETS = {
    'code_generation':  300,   # 60% — primary task, needs most volume
    'debugging':        150,   # 30% — was underrepresented (only 29 samples)
    'correction':       50,    # "no, that doesn't exist" — reduces hallucinated actions
    'full_application': 0,     # plan.md → complete app (deterministic, no target cap)
    'fim':              20,    # fill-in-the-middle (cheap, no LLM call)
    'syntax_qa':        20,    # Q&A from proposals
    'code_explanation': 20,    # explanation from real examples
    # translation: DISABLED — 2.5% success rate
    'architecture':     30,    # multi-feature-set apps with events and state
}
# Total: ~560 samples target

MAX_REPAIR_ATTEMPTS = 3   # retries per sample before discarding
_repair_rejects = []      # failed repair attempts → DPO negatives

with open(DATA_IN / 'knowledge.json') as f:
    kb = json.load(f)

# --- Load few-shot examples from knowledge base for generation prompts ---
# Pick 3 small, representative examples to include in code generation prompts.
_FEW_SHOT_EXAMPLES = []
_FEW_SHOT_PREFERRED = ['HelloWorld', 'Calculator', 'EventExample', 'StatusPost', 'OrderService']
for _ex in kb['examples']:
    if _ex['name'] in _FEW_SHOT_PREFERRED:
        _total = sum(len(v) for v in _ex.get('aro_files', {}).values())
        if _total < 3000:  # keep it short enough to fit in context
            _FEW_SHOT_EXAMPLES.append(_ex)
    if len(_FEW_SHOT_EXAMPLES) >= 3:
        break

def _format_few_shot_block():
    """Format few-shot examples as a reference block for generation prompts."""
    if not _FEW_SHOT_EXAMPLES:
        return ''
    parts = ['Here are examples of valid ARO code for reference:\n']
    for i, ex in enumerate(_FEW_SHOT_EXAMPLES, 1):
        parts.append(f'### Example {i}: {ex["name"]}')
        for fn, code in ex['aro_files'].items():
            parts.append(f'```aro\n{code.strip()}\n```')
        if ex.get('openapi'):
            parts.append(f'```yaml\n{ex["openapi"][:600].strip()}\n```')
        parts.append('')
    return '\n'.join(parts)

FEW_SHOT_BLOCK = _format_few_shot_block()

print(f'Knowledge base: {len(kb["actions"])} actions, {len(kb["examples"])} examples')
print(f'Few-shot examples loaded: {[e["name"] for e in _FEW_SHOT_EXAMPLES]}')
print(f'Targets: {sum(TARGETS.values())} total samples')

## Load model

In [ ]:
print(f'Loading {MODEL_ID}...')

# Auto-load warm-start adapter from notebook 04 if it exists.
# Notebook 04 writes the adapter path into knowledge.json after its fine-tune cell runs.
_warm_adapter = None
_kb_path = DATA_IN / 'knowledge.json'
if _kb_path.exists():
    try:
        with open(_kb_path) as _f:
            _kb_meta = json.load(_f)
        _warm_path = Path(_kb_meta.get('warm_start_adapter', ''))
        if _warm_path.exists() and (_warm_path / 'adapters.safetensors').exists():
            _warm_adapter = str(_warm_path)
            print(f'Found warm-start adapter: {_warm_adapter}')
    except Exception as _e:
        print(f'Could not read warm-start adapter path: {_e}')

if _warm_adapter:
    model, tokenizer = load(MODEL_ID, adapter_path=_warm_adapter)
    print(f'Model loaded with warm-start adapter (from notebook 04).')
else:
    model, tokenizer = load(MODEL_ID)
    print('Model loaded (base weights — run notebook 04 first for warm-start fine-tuning).')

# Propagate warm-start adapter into RL state so fine-tune rounds build on it
# (this variable is read by the RL config cell which also declares _rl_adapter)
_warm_start_adapter_path = Path(_warm_adapter) if _warm_adapter else None
print('Model ready.')

## System prompt (built from extracted action knowledge)

In [ ]:
# Build action reference dynamically from knowledge base
_action_lines = []
for _a in kb.get("actions", []):
    if _a.get("verbs"):
        _v = "/".join(_a["verbs"][:3])
        _r = _a.get("role", "OWN")
        _p = ", ".join(_a.get("prepositions", [])[:4])
        _d = _a.get("description", "")[:50]
        _action_lines.append(f"- {_v:<28} {_r:<10} {_p}")
action_ref = "\n".join(_action_lines)

ARO_SYSTEM_PROMPT = f"""You are an expert ARO language programmer.
ARO (Action Result Object) is a DSL for expressing business logic as natural-language statements.

SYNTAX:
  (FeatureSetName: BusinessActivity) {{
      Verb [the] <result[:qualifier]> preposition [the] <object[:qualifier]>.
  }}

KEY RULES:
- Articles (a/an/the) are optional everywhere
- String concatenation: ++ (NOT +)
- Variable names: hyphenated lowercase e.g. <user-id>, <order-total>
- Forbidden variable name prefixes: is-, with-, empty-
- Variables are IMMUTABLE — bind once; use a new name for each transformation
- Exactly ONE Application-Start per application (error if 0 or multiple)
- openapi.yaml REQUIRED for HTTP server; operationId must match feature set name
- Long-running apps: Keepalive the <application> for the <events>.

HTTP:
- Path params:   Extract the <id> from the <pathParameters: id>.
- Request body:  Extract the <data> from the <request: body>.
- Return statuses: <OK: status>, <Created: status>, <NoContent: status>,
                   <NotFound: status>, <BadRequest: status>, <Conflict: status>,
                   <Unauthorized: status>, <InternalServerError: status>

CONTROL FLOW:
- Conditional:   when <condition> {{ statements }}
- Loop:          For each <item> in <list> {{ statements }}
- Match:         match <var> {{ case value {{ statements }} case other {{ statements }} }}
- Guard on declaration: (Name: EventName Handler) when <field> = value {{ ... }}

COMPUTE & ARITHMETIC:
- Compute the <total> from <price> * <qty>.
- Compute the <upper: uppercase> from <text>.
- Compute the <len: length> from <text>.
- Supported ops: +, -, *, /, % (integers); ++ (strings)

CROSS-FEATURE SHARING:
- Publish as <alias> <variable>.  (makes variable accessible across feature sets in same business activity)

EVENTS:
- Emit a <EventName: event> with <data>.
- Handler:  (HandlerName: EventName Handler) {{ ... }}
- State:    Accept the <new-state: toState> for the <entity>.

LIFECYCLE:
- Application-Start: required entry point
- Application-End: Success (optional graceful shutdown)
- Application-End: Error (optional crash handler)

AVAILABLE ACTIONS (verb/aliases → role → prepositions):
{action_ref}


COMMON MISTAKES TO AVOID:
- Do NOT put expressions inside angle brackets: <price + tax> is WRONG. Use Compute first.
- Do NOT use == for comparison. Use Compare ... against or when <var> = value.
- Do NOT rebind variables. Create a NEW variable: <updated-count> not <count> again.
- The `where` clause is ONLY valid on Retrieve: Retrieve the <x> from <repo> where id = <id>.
  Do NOT use `where` on other actions like Filter — use Filter ... from <list> with <condition>.
- Each statement MUST end with a period (.)
- Angle brackets wrap ONLY identifiers: <user-id>, not <user.id> or <user+1>.
- Qualifiers use colon: <name: uppercase>, <id: pathParameters>, <OK: status>.

Always wrap ARO code in ```aro ... ``` fences.
Always generate complete, valid ARO that would pass `aro check`.
"""

print(f'System prompt: {len(ARO_SYSTEM_PROMPT)} chars')

# ── Build a strict verb whitelist from the knowledge base ─────────────────────
# The system prompt lists actions but the model still hallucinates verbs.
# This extracts the canonical first verb for each action from kb['actions']
# and appends a hard constraint to the system prompt.
_verb_list = sorted(set(
    a['verbs'][0] for a in kb.get('actions', []) if a.get('verbs')
))
VERB_HINT = (
    "\n\nIMPORTANT: Use ONLY these valid ARO action verbs: "
    + ", ".join(_verb_list)
    + "\nDo NOT invent new verbs. If you use an invalid verb, `aro check` will reject your code."
)
ARO_SYSTEM_PROMPT += VERB_HINT
print(f'Verb whitelist ({len(_verb_list)} verbs): {", ".join(_verb_list)}')
print(f'System prompt: {len(ARO_SYSTEM_PROMPT)} chars (with verb hint)')


## Core helpers

In [ ]:
def chat(messages, max_tokens=1500, temp=None):
    """Run the model on a message list and return the response string."""
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    sampler = make_sampler(temp=temp) if temp is not None else make_sampler()
    return mlx_generate(model, tokenizer, prompt=prompt, max_tokens=max_tokens, verbose=False, sampler=sampler)

def extract_aro_blocks(text):
    return [b.strip() for b in re.findall(r'```aro\n(.*?)```', text, re.DOTALL) if b.strip()]

def aro_check(code):
    """Returns (True/False/None, error_str). None = aro not available."""
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'check', tmp], capture_output=True, text=True, timeout=10)
            return r.returncode == 0, (r.stderr or r.stdout).strip()[:500]
    except FileNotFoundError:
        return None, 'aro_not_found'
    except subprocess.TimeoutExpired:
        return False, 'timeout'

def _is_comment_heavy(code):
    lines = [l.strip() for l in code.split('\n') if l.strip()]
    if not lines:
        return False
    comment_lines = sum(1 for l in lines if l.startswith('(*'))
    return comment_lines / len(lines) > 0.5

# ── Error-specific fix hints ─────────────────────────────────────────────────
# Maps error patterns to targeted advice the model can act on.
_ERROR_HINTS = [
    ('Expected preposition, but got =',
     'The `=` operator is not a preposition. Use `where id = <id>` ONLY with Retrieve. '
     'For comparisons, use: Compare the <result> against <value>. '
     'For conditions, use: when <var> = value {{ ... }}'),
    ('Expected preposition, but got {',
     'Curly braces are not allowed after a statement. '
     'when-guards go on the feature set header: (Name: Activity) when <cond> {{ ... }}'),
    ("Expected '>', but got +",
     'Do NOT put expressions inside angle brackets. '
     'WRONG: <price + tax>. RIGHT: Compute the <total> from <price> + <tax>.'),
    ("Expected '>', but got .",
     'Do NOT use dot notation inside angle brackets. '
     'WRONG: <user.name>. RIGHT: Extract the <name> from the <user: name>.'),
    ('Cannot rebind variable',
     'Variables are IMMUTABLE in ARO. You cannot reuse a variable name. '
     'Use a new name: <updated-count> instead of <count>.'),
    ("Expected '.', but got ==",
     'ARO does not use == for comparison. Each statement ends with a period. '
     'Use: Compare the <result> against <value>. Or: when <var> = value {{ ... }}'),
    ('Expected identifier, but got where',
     'The `where` clause is ONLY valid on Retrieve: '
     'Retrieve the <user> from the <repo> where id = <id>. '
     'Do NOT use `where` on Filter, Compute, or other actions.'),
    ('Expected action verb',
     'Each statement must start with a valid action verb (Extract, Compute, Log, Return, etc.). '
     'Check that every line inside {{ }} starts with a verb, not a variable or symbol.'),
    ('Entry point not found',
     'Every ARO application needs exactly one (Application-Start: Name) {{ ... }} feature set.'),
]

def _get_error_hint(error):
    """Return a targeted fix hint based on the error pattern."""
    for pattern, hint in _ERROR_HINTS:
        if pattern in error:
            return hint
    return ''


def generate_with_repair(instruction, max_tokens=1500, label='', skip_aro_check=False):
    """
    Generate ARO code with self-repair loop.

    On failure, each retry uses a FRESH repair prompt (not appended conversation)
    containing the broken code + error + targeted fix hint. Temperature decreases
    on each retry to make the model more conservative.

    Returns (output_str, attempts_used) or (None, MAX_REPAIR_ATTEMPTS) if all fail.
    """
    # Attempt 0: generate from the original instruction
    messages = [
        {'role': 'system', 'content': ARO_SYSTEM_PROMPT},
        {'role': 'user',   'content': instruction},
    ]
    temps = [None, 0.3, 0.1]  # decreasing temperature per retry

    last_code = None
    last_error = None

    for attempt in range(MAX_REPAIR_ATTEMPTS):
        tag = f'    [attempt {attempt+1}/{MAX_REPAIR_ATTEMPTS}]{(" " + label) if label else ""}'
        print(f'{tag} generating...', flush=True)

        temp_override = temps[attempt] if attempt < len(temps) else 0.1
        if attempt == 0:
            output = chat(messages, max_tokens=max_tokens)
        else:
            # Fresh repair prompt with broken code + error + hint
            hint = _get_error_hint(last_error or '')
            hint_str = f'\nHint: {hint}' if hint else ''

            repair_instruction = (
                f'The following ARO code has a syntax error:\n\n'
                f'```aro\n{last_code}\n```\n\n'
                f'Error from `aro check`:\n```\n{last_error}\n```\n'
                f'{hint_str}\n\n'
                f'Fix the code. Output ONLY the corrected ARO code in ```aro ... ``` fences. '
                f'Do NOT explain, just fix.'
            )
            repair_messages = [
                {'role': 'system', 'content': ARO_SYSTEM_PROMPT},
                {'role': 'user',   'content': repair_instruction},
            ]
            output = chat(repair_messages, max_tokens=max_tokens, temp=temp_override)

        preview = ' '.join(output.split())[:140]
        print(f'{tag} -> {preview}', flush=True)

        blocks = extract_aro_blocks(output)

        if not blocks:
            print(f'{tag} x no ```aro``` block -- retrying', flush=True)
            last_code = output[:500]
            last_error = 'No ```aro``` block found in output'
            continue

        code_text = '\n\n'.join(blocks)

        if _is_comment_heavy(code_text):
            print(f'{tag} x comment-heavy (>50% comment lines) -- retrying', flush=True)
            last_code = code_text
            last_error = 'More than 50% of lines are comments. Write executable ARO statements.'
            continue

        if skip_aro_check:
            print(f'{tag} ok (aro check skipped)', flush=True)
            return output, attempt

        passed, error = aro_check(code_text)

        if passed is True or passed is None:
            label_str = 'check passed' if passed is True else 'aro not available (accepted)'
            print(f'{tag} ok {label_str}', flush=True)
            return output, attempt

        print(f'{tag} x check failed: {error[:120]}', flush=True)
        last_code = code_text
        last_error = error

    print(f'    [discarded]{(" " + label) if label else ""} all {MAX_REPAIR_ATTEMPTS} attempts failed', flush=True)

    # Save all failed attempts as DPO negatives
    if last_code:
        _repair_rejects.append({
            'instruction': instruction,
            'output': last_code if '```aro' in (last_code or '') else f'```aro\n{last_code}\n```',
            'reason': 'repair_failed',
        })

    return None, MAX_REPAIR_ATTEMPTS


In [ ]:
def _show_sample(pairs, n=2, label=''):
    import random as _r
    sample_pool = _r.sample(pairs, min(n, len(pairs)))
    print(f'\n── Sample ({label}, {len(pairs)} total) ──────────────────────')
    for i, s in enumerate(sample_pool, 1):
        if 'messages' in s:
            user = s['messages'][1]['content'] if len(s['messages']) > 1 else ''
            asst = s['messages'][2]['content'] if len(s['messages']) > 2 else ''
        else:
            user = s.get('instruction', s.get('user', ''))
            asst = s.get('output', s.get('assistant', ''))
        task = s.get('task_type', s.get('source', '?'))
        print(f'  [{i}] task={task}')
        print(f'       USER: {user[:120].strip()!r}')
        print(f'       ASST: {asst[:120].strip()!r}')
    print('─' * 60)

## Output file + resume support

In [ ]:
OUTPUT_FILE = DATA_OUT / 'samples.jsonl'

def save(sample):
    with open(OUTPUT_FILE, 'a') as f:
        f.write(json.dumps(sample) + '\n')

# Resume: track which keys are already done
done = set()
if OUTPUT_FILE.exists():
    with open(OUTPUT_FILE) as f:
        for line in f:
            try:
                s = json.loads(line)
                done.add((s['task_type'], s.get('domain', s.get('source', s.get('scenario', s.get('heading', ''))))))
            except Exception:
                pass
    print(f'Resuming — {len(done)} samples already saved')

repair_stats = Counter()
_failed_prompts = []

## Reinforcement Learning: Explore & Learn

The model learns ARO by exploring — generating multiple candidates at varying temperatures,
validating each with `aro check` and `aro run`, and saving the ones that work as training data.
Every **30 valid samples**, a LoRA SFT round runs on all accumulated valid examples.
The improved adapter is hot-reloaded, so subsequent generation benefits immediately.

```
docs + proposals + examples
        │
        ▼
  explore K candidates          ← four temperatures per prompt
  (aro check + aro run)         ← real tool feedback, not proxy metrics
        │
  score ≥ 0.6?
   YES  ──→ save to samples.jsonl (becomes training data)
   NO   ──→ generate_with_repair fallback
        │
  every 30 valid samples
        ──→ LoRA SFT fine-tune on all saved samples
        ──→ hot-reload model with new adapter
        ──→ repeat (model gets better at ARO over time)
```

In [ ]:
# RL: Explore & Learn configuration
K_EXPLORE      = 3                       # 3 candidates — multi-sample exploration enabled
EXPLORE_TEMPS  = [0.3, 0.5, 0.7]         # 3 temps — diverse exploration
FINETUNE_EVERY = 999999                  # RL fine-tuning disabled — too slow for pipeline
LORA_LAYERS    = 8                       # LoRA adapter depth

PILOT_MODE     = False                   # Set True to generate only PILOT_SAMPLES, report pass rate, and stop
PILOT_SAMPLES  = 100

ADAPTER_DIR  = DATA_OUT.parent / 'adapters'
SFT_DATA_DIR = DATA_OUT.parent / 'sft_data'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
SFT_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Mutable state (updated by run_rl_finetune)
_rl_adapter        = None   # Path to current adapter, or None (base model)
_rl_round          = 0      # Fine-tune round counter
_samples_since_tune = 0     # New valid samples since last fine-tune
_rl_losses         = {}     # {round: {train_iters, train_losses, val_iters, val_losses}}

# Inherit warm-start adapter as the starting point for RL rounds
if '_warm_start_adapter_path' in dir() and _warm_start_adapter_path:
    _rl_adapter = _warm_start_adapter_path
    print(f'RL starting from warm-start adapter: {_rl_adapter}')

print(f'RL config: K={K_EXPLORE} temps={EXPLORE_TEMPS} finetune_every={FINETUNE_EVERY}')
if PILOT_MODE:
    print(f'PILOT MODE enabled: will generate at most {PILOT_SAMPLES} samples, then report pass rate and stop')


In [ ]:
def aro_run_validate(code, timeout=10):
    """
    Run code with `aro run` and return (ok, output_str).
      True  = ran cleanly
      True  = timed out (likely a server app — that's valid)
      False = crashed with non-zero exit
      None  = aro binary not found
    """
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(
                ['aro', 'run', tmp],
                capture_output=True, text=True, timeout=timeout,
            )
            out = (r.stdout + r.stderr).strip()
            return (r.returncode == 0), out[:300]
    except FileNotFoundError:
        return None, 'aro not found'
    except subprocess.TimeoutExpired:
        return True, 'server_timeout'


def score_candidate(raw_output):
    """
    Score a raw model output using aro check + aro run.
    Returns (score: float, detail: str).

      0.0  no ARO block at all
      0.1  has ARO block but aro check failed
      0.5  aro not available (neutral — structural signal only)
      0.8  aro check passed; run skipped (server/Keepalive app)
      0.9  aro check passed; run unavailable
      1.0  aro check passed AND aro run succeeded
    """
    blocks = extract_aro_blocks(raw_output)
    if not blocks:
        return 0.0, 'no_aro_block'

    code = '\n\n'.join(blocks)
    check_ok, check_err = aro_check(code)

    if check_ok is None:
        return 0.5, 'aro_unavailable'
    if not check_ok:
        short = check_err.split('\n')[0][:80]
        return 0.1, f'check_failed: {short}'

    # Check passed — skip run for long-running server apps
    if 'Keepalive' in code or 'http-server' in code:
        return 0.8, 'check_passed (server — run skipped)'

    run_ok, run_out = aro_run_validate(code)
    if run_ok is None:
        return 0.9, 'check_passed (run unavailable)'
    if run_ok:
        short = run_out.split('\n')[0][:60]
        return 1.0, f'check+run passed: {short}'
    short = run_out.split('\n')[0][:60]
    return 0.8, f'check_passed run_failed: {short}'

print('Validation functions ready.')


In [ ]:
from mlx_lm.sample_utils import make_sampler


def explore_candidates(instruction, max_tokens=1500):
    """
    GRPO-style exploration: sample K candidates at different temperatures.
    Returns list of (output, score, detail) sorted best-first.
    The best-scoring candidate that passes aro check becomes training data.
    """
    messages = [
        {'role': 'system', 'content': ARO_SYSTEM_PROMPT},
        {'role': 'user',   'content': instruction},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    results = []
    for i, temp in enumerate(EXPLORE_TEMPS[:K_EXPLORE]):
        print(f'    [explore {i+1}/{K_EXPLORE} T={temp:.1f}] generating...', flush=True)
        output = mlx_generate(
            model, tokenizer, prompt=prompt,
            max_tokens=max_tokens, verbose=False,
            sampler=make_sampler(temp=temp),
        )
        score, detail = score_candidate(output)
        preview = ' '.join(output.split())[:130]
        print(f'    [explore {i+1}] score={score:.1f}  {detail}', flush=True)
        print(f'    [explore {i+1}] → {preview}', flush=True)
        results.append((output, score, detail))

    results.sort(key=lambda x: -x[1])
    return results


In [ ]:
def run_rl_finetune():
    """
    Fine-tune on all valid samples accumulated in samples.jsonl (LoRA SFT).
    Frees the model from memory, runs mlx_lm lora in a subprocess,
    then hot-reloads with the new adapter so subsequent generation benefits.
    """
    global model, tokenizer, _rl_adapter, _rl_round, _rl_losses

    _rl_round += 1
    adapter_out = ADAPTER_DIR / f'round_{_rl_round:03d}'

    # Build SFT chat-format data from every valid sample saved so far
    samples = []
    if OUTPUT_FILE.exists():
        with open(OUTPUT_FILE) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                s = json.loads(line)
                instr  = s.get('instruction', '')
                output = s.get('output', '')
                if instr and output:
                    samples.append({'messages': [
                        {'role': 'system',    'content': ARO_SYSTEM_PROMPT},
                        {'role': 'user',      'content': instr},
                        {'role': 'assistant', 'content': output},
                    ]})

    if len(samples) < 4:
        print(f'  [RL] Skipping fine-tune — only {len(samples)} samples (need ≥ 4)', flush=True)
        return

    random.shuffle(samples)
    split = max(1, int(len(samples) * 0.1))

    SFT_DATA_DIR.mkdir(parents=True, exist_ok=True)
    (SFT_DATA_DIR / 'valid.jsonl').write_text(
        '\n'.join(json.dumps(s) for s in samples[:split]))
    (SFT_DATA_DIR / 'train.jsonl').write_text(
        '\n'.join(json.dumps(s) for s in samples[split:]))

    # Scale steps: roughly 2 passes over the data, min 50 max 200
    iters = max(50, min(200, len(samples) * 2))

    cmd = [
        sys.executable, '-m', 'mlx_lm', 'lora',
        '--model',         MODEL_ID,
        '--data',          str(SFT_DATA_DIR),
        '--train',
        '--num-layers',    str(LORA_LAYERS),
        '--iters',         str(iters),
        '--batch-size',    '2',
        '--learning-rate', '2e-4',
        '--adapter-path',  str(adapter_out),
        '--mask-prompt',
    ]
    if _rl_adapter:
        cmd += ['--resume-adapter-file',
                str(_rl_adapter / 'adapters.safetensors')]

    print(f'\n  [RL round {_rl_round}] fine-tuning on {len(samples)} samples, {iters} steps...', flush=True)

    # Free unified memory before the subprocess training run
    del model, tokenizer
    import gc, re as _re
    gc.collect()

    # Stream output in real-time — avoids timeout on large models; also parses losses
    _loss_re = _re.compile(r'Iter\s+(\d+):\s+(Train|Val) loss\s+([\d.]+)')
    round_losses = {'train_iters': [], 'train_losses': [], 'val_iters': [], 'val_losses': []}

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(f'  [RL] {line}', end='', flush=True)
        m = _loss_re.search(line)
        if m:
            it, kind, val = int(m.group(1)), m.group(2), float(m.group(3))
            if kind == 'Train':
                round_losses['train_iters'].append(it)
                round_losses['train_losses'].append(val)
            else:
                round_losses['val_iters'].append(it)
                round_losses['val_losses'].append(val)
    returncode = proc.wait()

    _rl_losses[_rl_round] = round_losses

    if returncode == 0:
        _rl_adapter = adapter_out
        print(f'  [RL] Training complete → {adapter_out}', flush=True)
    else:
        print(f'  [RL] Training failed (round {_rl_round}) with exit code {returncode}', flush=True)
        # Fall back to last known good adapter (or base)

    # Hot-reload: base model + adapter fused in-memory (no disk write needed)
    load_kwargs = {'adapter_path': str(_rl_adapter)} if _rl_adapter else {}
    model, tokenizer = load(MODEL_ID, **load_kwargs)
    print(f'  [RL] Model reloaded'
          + (f' with adapter (round {_rl_round})' if _rl_adapter else ' (base weights)'),
          flush=True)


In [ ]:
# ── Grounded domains: load (instruction, reference_code) from real Examples ──
# Produced by notebook 03 Phase 1. Each entry has a known-correct ARO solution,
# so we can measure how close the model gets and always save the reference as
# ground-truth training data regardless of what the model generates.

GROUNDED_DOMAINS = []

_kp_file = DATA_IN / 'knowledge_pairs.jsonl'
if _kp_file.exists():
    with open(_kp_file) as _f:
        for _line in _f:
            if not _line.strip():
                continue
            try:
                _p = json.loads(_line)
                if _p.get('source', '').startswith('example:'):
                    GROUNDED_DOMAINS.append({
                        'instruction': _p.get('instruction', next((m['content'] for m in _p.get('messages',[]) if m.get('role')=='user'), '')),
                        'reference':   _p.get('output', _p.get('messages',[{}]*3)[2].get('content','')),  # real, validated ARO code
                        'name':        _p.get('source', '').split(':', 1)[1],
                    })
            except Exception:
                pass
    print(f'Loaded {len(GROUNDED_DOMAINS)} grounded domains from real examples')
    print('These are interleaved 1:3 with synthetic domains in Task 1.')
    print('For each, the reference solution is always saved as score=1.0 training data.')
else:
    print('No knowledge_pairs.jsonl found — run notebooks 03 and 04 first.')
    print('Task 1 will use synthetic domains only.')


def _feature_set_names(code):
    """Extract the set of feature-set names from ARO code."""
    return set(re.findall(r'\(\s*([^:)]+?)\s*:', code))


def _reference_similarity(generated_output, reference_output):
    """
    Fraction of reference feature-set names reproduced in the generated output.
    1.0 = all reference feature sets present, 0.0 = none.
    """
    ref_names = _feature_set_names(reference_output)
    gen_names = _feature_set_names(generated_output)
    if not ref_names:
        return 0.5
    return round(len(ref_names & gen_names) / len(ref_names), 3)


def _grounded_prompt(domain_info):
    """
    Build a reference-guided instruction: the model sees the real example
    as a pattern guide before being asked to write the new application.
    """
    ref_preview = domain_info['reference'][:2000]
    return (
        f'{domain_info["instruction"]}\n\n'
        f'Study this working ARO example to learn the correct patterns:\n\n'
        f'{ref_preview}\n\n'
        f'Now write the complete ARO application described in the instruction above, '
        f'following the same structural patterns.'
    )


## Task 1 — Code Generation

In [ ]:
DOMAINS = [
    # ── Web APIs & CRUD ──────────────────────────────────────────────────────
    'todo list API with create, list, complete, and delete',
    'product catalog API with search and pagination',
    'blog posts API with comments and draft/publish workflow',
    'task manager API with priorities and due dates',
    'event registration API with capacity limits',
    'customer orders API with status state machine',
    'employee directory API with department filtering',
    'support ticket API with severity levels',
    'appointment scheduling API with conflict detection',
    'notification preferences API per user',
    'poll and voting API with results',
    'book library API with borrowing and returns',
    'user profiles API with avatar upload',
    'subscription billing API with renewal alerts',
    'project milestones API and task tracking',
    'contact list API with tags and search',
    'feature flag manager API with rollout percentage',
    'audit log viewer API with filtering',
    'API key management with rate limits',
    'webhook delivery API with retry logic',
    'document versioning API with diff',
    'team management API with roles',
    'coupon codes API with expiry and usage limits',
    'recipe collection API with ingredient search',
    'inventory tracking API with low-stock email alerts',

    # ── File & Data Operations ───────────────────────────────────────────────
    'CSV file reader that parses rows and computes column statistics',
    'log file tail watcher that monitors a directory for new .log files',
    'JSON config merger that reads multiple config files and deep-merges them',
    'file deduplicator that hashes files in a directory and reports duplicates',
    'backup rotator that copies files to a dated archive folder and prunes old backups',
    'YAML validator that reads .yaml files from a directory and reports parse errors',
    'markdown link checker that reads .md files and validates internal links exist',
    'CSV to JSON converter that reads a CSV file and writes each row as a JSON object',
    'file metadata reporter that lists files with size, modified date, and permissions',
    'directory tree printer that recursively lists a folder structure with indentation',

    # ── Data Engineering & Pipelines ─────────────────────────────────────────
    'data pipeline that reads JSON records, filters by field value, and writes results',
    'ETL job that extracts data from an API, transforms fields, and stores in repository',
    'aggregation service that groups records by category and computes sum and average',
    'map-reduce pipeline that splits a large dataset, processes chunks, and merges results',
    'data quality checker that validates records against a schema and emits error events',
    'time series aggregator that buckets events by hour and computes counts per bucket',
    'deduplication pipeline that reads records, detects duplicates by key, and keeps latest',
    'data enrichment service that fetches external API data and merges into local records',
    'batch processor that reads a file line by line, transforms each, and writes output',
    'streaming word counter that reads text files and computes word frequency sorted by count',

    # ── DevOps & Deployment ──────────────────────────────────────────────────
    'deployment status tracker with rollback state machine (pending, deploying, live, rolled-back)',
    'health check monitor that polls endpoints and emits alerts on failure',
    'service registry that tracks running services with heartbeat expiry',
    'release notes generator that reads git commit messages and formats a changelog',
    'environment config manager that reads env-specific YAML and validates required keys',
    'container restart watcher that monitors a socket for crash events and logs restarts',
    'canary deployment controller with traffic percentage and automatic rollback on errors',
    'secret rotation scheduler that tracks expiry dates and emits renewal events',
    'incident tracker with severity levels and escalation state machine',
    'uptime dashboard that stores check results and computes availability percentage',

    # ── File System Watching & Automation ────────────────────────────────────
    'file watcher that monitors a directory and emits events on create, modify, and delete',
    'hot-reload config service that watches a YAML file and re-applies settings on change',
    'image thumbnail generator that watches an upload folder and creates resized copies',
    'log rotator that watches log file size and archives when threshold is exceeded',
    'sync service that watches a source directory and mirrors changes to a destination',
    'file classifier that watches an inbox folder and moves files to subfolders by extension',
    'template renderer that watches .mustache files and re-renders HTML on change',

    # ── CLI Tools & Utilities ────────────────────────────────────────────────
    'command-line calculator that parses arithmetic expressions from parameters',
    'password generator that accepts length and charset options and outputs random strings',
    'JSON pretty printer that reads a file path and outputs formatted JSON',
    'port scanner that checks a range of TCP ports on a target host',
    'base64 encoder/decoder that reads a file and outputs encoded or decoded content',
    'UUID generator that creates batch UUIDs and writes them to a file',
    'string diff tool that reads two files and reports added, removed, and changed lines',
    'CSV column extractor that reads a CSV and outputs only specified columns',
    'hash checker that computes SHA256 of files in a directory and compares to a manifest',
    'timestamp converter that accepts unix epoch and outputs human-readable date formats',

    # ── Monitoring & Metrics ─────────────────────────────────────────────────
    'Prometheus metrics exporter for request counts, latency histograms, and error rates',
    'disk usage monitor that checks mount points and emits alerts above threshold',
    'process watchdog that monitors a PID file and restarts the service on crash',
    'SLA tracker that computes uptime percentage from health check events',
    'request rate limiter that tracks per-client request counts with sliding window',

    # ── Event-Driven & Messaging ─────────────────────────────────────────────
    'order fulfillment pipeline: order placed, payment verified, shipped, delivered',
    'email notification hub that listens for domain events and sends templated emails',
    'chat message relay that receives socket messages and broadcasts to connected clients',
    'event sourcing ledger that stores every state change as an immutable event',
    'retry queue manager that re-emits failed events with exponential backoff',
    'multi-service orchestrator that coordinates two external API calls and merges results',

    # ── Domain-Specific Applications ─────────────────────────────────────────
    'weather dashboard that fetches forecast API data and formats a daily summary',
    'expense tracker with receipt upload, categorization, and monthly totals',
    'fitness log API with exercises, sets, reps, and weekly progress summary',
    'plant watering scheduler that tracks plants, watering intervals, and overdue alerts',
    'pet feeding tracker with multiple pets, meal schedules, and low-food alerts',
    'home energy monitor that reads smart meter data and computes daily cost',
    'meeting room booker with time slots, conflict detection, and cancellation',
    'quiz engine that serves questions, tracks scores, and returns results',

    # ── Inspired by ARO-Application projects ─────────────────────────────────
    # From: Buildflow, Crawler, FediTerm, MastoStream, StatusPost, Sumup,
    #       SystemLoadMonitor, mm, mostrecentfile, rulpz, sf
    'CI/CD build pipeline orchestrator with step sequencing and status state machine',
    'web crawler that fetches pages, extracts links, normalizes URLs, and stores results',
    'Mastodon/Fediverse terminal client with timeline display and post composing',
    'social media streaming client that connects to SSE feed and renders updates',
    'real-time message board with WebSocket broadcasts and message repository',
    'arithmetic expression evaluator that parses and computes nested expressions',
    'system load monitor that reads CPU and memory metrics and displays in terminal UI',
    'media metadata extractor that scans directories and reports file properties',
    'most-recent-file finder that scans directories and displays sorted results',
    'social network API with user auth, posts, timelines, and follow relationships',
    'file search utility that recursively finds files matching glob patterns',

    # ── Inspired by Examples/ directory ──────────────────────────────────────
    # From: AssertDemo, AutoPipeline, CollectionMerge, ConstantFolding,
    #       ContextAware, CountingStyles, DateRangeDemo, EchoSocket,
    #       EventReplay, ExtractInCase, FormatAwareIO, HashTest,
    #       Immutability, LinkHeaderPagination, ListTest, MetricsDemo,
    #       ModulesExample, MultiService, ObjectUpdate, ParallelForEach,
    #       PipelineDemo, RawStrings, RepositoryLimits, RepositoryObserver,
    #       SetOperations, SinkSyntax, SortExample, SSEClient, SSEStreamDemo,
    #       StateMachine, StreamingDemo, StreamingPipeline, TerminalUI
    'assertion and validation demo that tests expected values with Assert action',
    'automatic pipeline detection that chains filter, transform, aggregate without explicit pipes',
    'collection merge utility that combines objects and lists with Merge action',
    'context-aware formatter that adapts output for console, HTTP, and debug contexts',
    'date range calculator with recurrence patterns and date arithmetic',
    'TCP echo server that accepts connections and echoes back received messages',
    'event replay system that records events and replays them for testing',
    'format-aware file I/O that auto-detects JSON, YAML, and CSV formats',
    'immutability demo showing new-name pattern and qualifier-as-name for transforms',
    'link header pagination client that follows next/prev links automatically',
    'parallel for-each processor that runs iterations concurrently',
    'repository observer that watches data changes and triggers side effects',
    'set operations demo with union, intersect, and difference on collections',
    'state machine with Accept action for order lifecycle transitions',
    'streaming pipeline with lazy evaluation and tee for fan-out processing',
    'SSE (Server-Sent Events) client that connects to an event stream',
    'terminal UI application with screen templates and keyboard navigation',
    'multi-service application combining HTTP server, file watcher, and socket server',
    'raw string demo with heredoc-style multiline string literals',
    'sort utility that orders collections by field with ascending/descending',
    'modules example with cross-module communication via events',

    # ── WebSocket & Real-Time ────────────────────────────────────────────────
    'WebSocket chat room with join/leave events and message broadcasting',
    'real-time stock ticker that streams price updates via WebSocket',
    'collaborative editor backend that syncs document changes via WebSocket',
    'live dashboard that pushes metric updates to connected WebSocket clients',

    # ── Template & Rendering ─────────────────────────────────────────────────
    'email template renderer that loads mustache templates and fills user data',
    'report generator that reads data from repository and renders HTML template',
    'invoice generator that computes line totals and renders a formatted invoice',

    # ── Testing & Quality ────────────────────────────────────────────────────
    'test suite runner that executes Given/When/Then test cases and reports results',
    'API contract validator that checks responses against OpenAPI schema',
    'data integrity checker that validates repository contents against constraints',
]

# ── Detect domain type for instruction template ──────────────────────────────
_API_KEYWORDS = {'API', 'api', 'endpoint', 'REST', 'CRUD', 'webhook'}
_FILE_KEYWORDS = {'file', 'CSV', 'JSON', 'YAML', 'directory', 'folder', 'reads', 'writes', 'watcher'}
_EVENT_KEYWORDS = {'event', 'pipeline', 'state machine', 'emits', 'listens'}

def _make_instruction(domain):
    """Generate an appropriate instruction based on the domain type.
    Includes few-shot examples from the knowledge base for reference."""
    d_upper = domain.upper()
    is_api = any(kw in domain for kw in _API_KEYWORDS)
    is_file = any(kw in domain for kw in _FILE_KEYWORDS)
    is_event = any(kw in domain for kw in _EVENT_KEYWORDS)

    # Prepend few-shot examples so the model sees valid ARO patterns
    few_shot = FEW_SHOT_BLOCK + '\n\n' if FEW_SHOT_BLOCK else ''

    if is_api:
        return (
            f'{few_shot}'
            f'Write a complete ARO HTTP API application for: {domain}.\n\n'
            f'Include:\n'
            f'1. openapi.yaml with at least 3 endpoints\n'
            f'2. main.aro with feature sets matching every operationId\n'
            f'3. An Application-Start feature set\n\n'
            f'Label each file:\n## openapi.yaml\n```yaml\n...\n```\n\n'
            f'## main.aro\n```aro\n...\n```'
        )
    elif is_file:
        return (
            f'{few_shot}'
            f'Write a complete ARO application for: {domain}.\n\n'
            f'Include:\n'
            f'1. main.aro with Application-Start that sets up file monitoring or reads input\n'
            f'2. Feature sets for each processing step\n'
            f'3. Use Read/Write/Copy/Move file actions and file event handlers where appropriate\n\n'
            f'## main.aro\n```aro\n...\n```'
        )
    elif is_event:
        return (
            f'{few_shot}'
            f'Write a complete ARO event-driven application for: {domain}.\n\n'
            f'Include:\n'
            f'1. main.aro with Application-Start\n'
            f'2. Feature sets that Emit domain events and Handler feature sets that react to them\n'
            f'3. Use Accept for state transitions if applicable\n\n'
            f'## main.aro\n```aro\n...\n```'
        )
    else:
        return (
            f'{few_shot}'
            f'Write a complete ARO application for: {domain}.\n\n'
            f'Include:\n'
            f'1. main.aro with Application-Start\n'
            f'2. Feature sets for each operation\n'
            f'3. Use appropriate ARO actions (Compute, Store, Retrieve, Log, etc.)\n\n'
            f'## main.aro\n```aro\n...\n```'
        )


# ── Pilot mode: cap all targets to test before committing to full run ─────
if PILOT_MODE:
    _pilot_total = 0
    for _k in TARGETS:
        TARGETS[_k] = min(TARGETS[_k], max(1, PILOT_SAMPLES * TARGETS[_k] // max(1, sum(TARGETS.values()))))
        _pilot_total += TARGETS[_k]
    print(f'PILOT MODE: capped targets to {_pilot_total} total samples (from {PILOT_SAMPLES} budget)')
    print(f'  Adjusted targets: {TARGETS}')

print(f'\n--- Task 1 (RL Explore & Learn): Code Generation (target={TARGETS["code_generation"]}) ---')
print(f'    Synthetic domains : {len(DOMAINS)}')
print(f'    Grounded domains  : {len(GROUNDED_DOMAINS)} (interleaved 1 per 3 synthetic)')

count = 0
_samples_since_tune = 0

# ── Build an interleaved stream: synthetic x 3, then grounded x 1 ──────────
_grounded_pool = GROUNDED_DOMAINS.copy()
random.shuffle(_grounded_pool)
_synthetic_pool = (DOMAINS * 10)[:TARGETS['code_generation'] * 2]

def _domain_stream():
    """Yield (instruction, reference|None, label, is_grounded)."""
    g_iter = iter(_grounded_pool * 10)
    s_iter = iter(_synthetic_pool)
    since_grounded = 0
    while True:
        # Every 3 synthetic items, emit one grounded item (if available)
        if since_grounded >= 3 and _grounded_pool:
            try:
                g = next(g_iter)
                since_grounded = 0
                yield _grounded_prompt(g), g['reference'], g['name'], True
                continue
            except StopIteration:
                pass
        try:
            d = next(s_iter)
            since_grounded += 1
            instr = _make_instruction(d)
            yield instr, None, d, False
        except StopIteration:
            break

for instruction, reference, label, is_grounded in _domain_stream():
    if count >= TARGETS['code_generation']:
        break

    key = ('code_generation', f'{"grounded:" if is_grounded else ""}{label}')
    if key in done:
        continue

    tag = '[G]' if is_grounded else '   '
    print(f'\n{tag} [{count+1}] {label[:70]}', flush=True)

    # ── GRPO exploration ─────────────────────────────────────────────────────
    results = explore_candidates(instruction)
    best_output, best_score, best_detail = results[0]

    if best_score >= 0.6:
        output   = best_output
        attempts = 0
        if is_grounded and reference:
            sim = _reference_similarity(best_output, reference)
            print(f'    accepted (score={best_score:.1f}, ref_sim={sim:.2f}: {best_detail})', flush=True)
        else:
            print(f'    accepted (score={best_score:.1f}: {best_detail})', flush=True)
    else:
        print(f'    repair loop (best={best_score:.1f}: {best_detail})', flush=True)
        output, attempts = generate_with_repair(instruction, label=label[:50])

    repair_stats[attempts] += 1

    if not output or not extract_aro_blocks(output):
        _failed_prompts.append({'instruction': label, 'reason': 'no_output' if not output else 'no_aro_blocks', 'grounded': is_grounded})
        continue

    if output and extract_aro_blocks(output):
        # Comment-density filter for explore_candidates path (repair path already filtered)
        blocks = extract_aro_blocks(output)
        if _is_comment_heavy('\n\n'.join(blocks)):
            _failed_prompts.append({'instruction': label, 'reason': 'comment_heavy', 'grounded': is_grounded})
            print(f'    skipped (>50% comment lines)', flush=True)
            continue

        # Save the generated sample
        save({
            'task_type':    'code_generation',
            'instruction':  label,
            'output':       output,
            'domain':       label,
            'score':        best_score,
            'grounded':     is_grounded,
        })
        done.add(key)
        count += 1
        _samples_since_tune += 1
        print(f'  [{count}] saved (score={best_score:.1f}){" [G]" if is_grounded else ""} {label[:60]}', flush=True)

        # ── For grounded domains: ALSO save the reference as score=1.0 ──────
        # The real example is always correct. Saving it gives the model a direct
        # "ground truth" answer for this exact domain.
        if is_grounded and reference:
            ref_key = ('code_generation', f'reference:{label}')
            if ref_key not in done:
                save({
                    'task_type':    'code_generation',
                    'instruction':  label,
                    'output':       reference,
                    'domain':       label,
                    'score':        1.0,
                    'grounded':     True,
                    'is_reference': True,
                })
                done.add(ref_key)
                print(f'         + reference saved (score=1.0) {label[:55]}', flush=True)

        # ── Trigger LoRA fine-tune ───────────────────────────────────────────
        if _samples_since_tune >= FINETUNE_EVERY:
            run_rl_finetune()
            _samples_since_tune = 0

print(f'\nDone: {count} samples')
print(f'Failed prompts: {len(_failed_prompts)}')
print(f'RL rounds completed: {_rl_round}')


In [ ]:
# Show sample of generated code_generation pairs
_cg_pairs = []
with open(OUTPUT_FILE) as _f:
    for _line in _f:
        if _line.strip():
            _s = json.loads(_line)
            if _s.get('task_type') == 'code_generation':
                _cg_pairs.append(_s)
_show_sample(_cg_pairs, label='NB10 code_generation')

# ── Dump failed prompts to GEN_TODO.md for later training data improvement ───
_todo_path = ARO_ROOT / 'GEN_TODO.md'
if _failed_prompts:
    with open(_todo_path, 'w') as _f:
        _f.write('# Generation TODO\n\n')
        _f.write(f'Failed prompts from NB10 code generation ({len(_failed_prompts)} total).\n')
        _f.write('These prompts did not produce valid ARO output. Investigate and add\n')
        _f.write('manual training pairs or adjust the prompts for the next run.\n\n')
        for i, fp in enumerate(_failed_prompts, 1):
            reason = fp['reason']
            grounded = ' [grounded]' if fp.get('grounded') else ''
            _f.write(f'## {i}. {reason}{grounded}\n\n')
            _f.write(f'{fp["instruction"]}\n\n')
            _f.write('---\n\n')
    print(f'Wrote {len(_failed_prompts)} failed prompts to {_todo_path}')
else:
    # Clear any stale GEN_TODO.md from a previous run
    if _todo_path.exists():
        _todo_path.unlink()
    print('No failed prompts — GEN_TODO.md not needed')


## Task 2 — Code Explanation

In [ ]:
print(f'\n--- Task 2: Code Explanation (target={TARGETS["code_explanation"]}) ---')
examples = kb['examples']
count = 0
for ex in random.sample(examples * 5, min(TARGETS['code_explanation'] * 3, len(examples) * 5)):
    if count >= TARGETS['code_explanation']:
        break
    key = ('code_explanation', ex['name'])
    if key in done:
        continue
    code = '\n\n'.join(f'// {fn}\n{c}' for fn, c in ex['aro_files'].items())
    if len(code.strip()) < 80:
        continue
    user_prompt = f"""You are explaining ARO code to a developer learning the language.

```aro
{code[:2500]}
```

Write a clear explanation covering:
1. **Purpose**: What does this application do? (1-2 sentences)
2. **Entry point**: How does it start? What services does it initialize?
3. **Event flow**: What events are handled and in what order?
4. **Data flow**: What data is extracted, transformed, and returned?
5. **Key design**: Any notable patterns (state machine, observer, HTTP API, etc.)?

Write in plain English. Keep it under 200 words. Do NOT generate new ARO code."""
    messages = [
        {'role': 'system', 'content': ARO_SYSTEM_PROMPT},
        {'role': 'user',   'content': user_prompt},
    ]
    output = chat(messages, max_tokens=600)
    save({'task_type': 'code_explanation', 'instruction': 'Explain what this ARO application does, focusing on the event flow and data movement.', 'input': code[:2500], 'output': output, 'source': ex['name']})
    done.add(key)
    count += 1
    print(f'  [{count}] {ex["name"]}')

print(f'Done: {count} samples')

In [ ]:
# Show sample of generated code_explanation pairs
_ce_pairs = []
with open(OUTPUT_FILE) as _f:
    for _line in _f:
        if _line.strip():
            _s = json.loads(_line)
            if _s.get('task_type') == 'code_explanation':
                _ce_pairs.append(_s)
_show_sample(_ce_pairs, label='NB10 code_explanation')

## Task 3 — Debugging

In [ ]:
# Bug types derived from real git fix history in ARO-Application repos.
# Each tuple: (bug_id, mutation_instruction, typical_aro_check_error)
# The error message is used as part of the user prompt for error->fix training.

BUG_TYPES = [
    # ── Preposition & syntax errors (most common in real repos) ──────────────
    ('wrong_preposition_retrieve',
     'Change the preposition in a Retrieve statement from `from` to `with`',
     'Retrieve expects preposition `from` or `where`, got `with`'),
    ('wrong_preposition_extract',
     'Change the preposition in an Extract statement from `from` to `for`',
     'Extract expects preposition `from`, got `for`'),
    ('wrong_preposition_store',
     'Change the preposition in a Store statement from `into` to `from`',
     'Store expects preposition `into`, got `from`'),
    ('wrong_preposition_return',
     'Change the preposition in a Return statement from `for` to `from`',
     'Return expects preposition `with` or `for`, got `from`'),
    ('wrong_preposition_emit',
     'Change the preposition in an Emit statement from `with` to `from`',
     'Emit expects preposition `with`, got `from`'),
    ('wrong_preposition_log',
     'Change the preposition in a Log statement from `to` to `with`',
     'Log expects preposition `to`, got `with`'),

    # ── String concatenation (++ vs +) -- very frequent real-world mistake ────
    ('string_concat_plus',
     'Replace one `++` with `+` for string concatenation',
     'Cannot apply arithmetic + to string operands; use ++ for concatenation'),
    ('string_concat_dot',
     'Replace one `++` with `.` (Python-style) for string concatenation',
     'Unexpected token `.` in expression; use ++ for string concatenation'),

    # ── Variable naming violations ───────────────────────────────────────────
    ('reserved_prefix_is',
     'Rename one variable to start with `is-` (e.g. `<is-valid>`)',
     'error: Expected identifier, but got is -- reserved prefix'),
    ('reserved_prefix_with',
     'Rename one variable to start with `with-` (e.g. `<with-data>`)',
     'error: Expected identifier, but got with -- reserved prefix'),
    ('reserved_prefix_empty',
     'Rename one variable to start with `empty-` (e.g. `<empty-list>`)',
     'error: Expected identifier, but got empty -- reserved prefix'),
    ('camelCase_variable',
     'Rename one hyphenated variable to camelCase (e.g. `<user-id>` -> `<userId>`)',
     'warning: Variable name should use hyphenated-lowercase, not camelCase'),

    # ── Application lifecycle bugs ───────────────────────────────────────────
    ('missing_app_start',
     'Remove the Application-Start feature set entirely',
     'error: No Application-Start found -- exactly one is required per application'),
    ('duplicate_app_start',
     'Add a second Application-Start feature set with a different business activity name',
     'error: Multiple Application-Start feature sets found -- only one allowed'),
    ('missing_return',
     'Remove the Return statement from one feature set',
     'warning: Feature set has no Return statement -- execution falls through'),

    # ── HTTP / OpenAPI mismatches ────────────────────────────────────────────
    ('operationId_typo',
     'Add a typo to one feature set name so it no longer matches its operationId',
     'warning: Feature set name does not match any operationId in openapi.yaml'),
    ('wrong_status_code',
     'Change `Created: status` to `OK: status` on a POST endpoint return',
     'Semantically incorrect: POST create should return Created, not OK'),
    ('missing_path_param',
     'Remove the `Extract the <id> from the <pathParameters: id>` line in a GET-by-id handler',
     'warning: Path parameter `id` declared in OpenAPI but never extracted'),

    # ── Delete syntax (real bug from sf repo) ────────────────────────────────
    ('delete_missing_result',
     'Change `Delete the <all> from the <repository>` to `Delete from the <repository>` (omit result)',
     'error: Expected `<`, but got preposition(from) -- Delete requires a result identifier'),

    # ── Qualifier placement (real bug from mm repo) ──────────────────────────
    ('qualifier_wrong_side',
     'Move a qualifier from result to object side: e.g. `Extract the <name: last> from <parts>` -> `Extract the <name> from <parts: last>`',
     'Qualifier on object side is treated as field path, not list accessor -- move to result side'),

    # ── Immutability violations ──────────────────────────────────────────────
    ('reuse_variable_name',
     'Reuse the same variable name for a second binding: e.g. bind `<total>` twice with different Compute statements',
     'error: Variable `total` is already bound -- variables are immutable, use a new name'),
    ('update_without_new_name',
     'Use `Compute the <count> from <count> + 1` to mutate a variable in place',
     'error: Cannot rebind `count` -- use a new name like `<new-count>`'),

    # ── Event handling bugs ──────────────────────────────────────────────────
    ('emit_wrong_event_name',
     'Change the event name in an Emit statement so it does not match any Handler declaration',
     'warning: Event `OrderPlaced` is emitted but no handler matches'),
    ('handler_missing_extract',
     'Remove the `Extract the <data> from the <event: ...>` line in an event handler',
     'warning: Handler does not extract any data from the event'),
    ('observer_wrong_repo',
     'Change the repository name in a Repository Observer so it does not match any Store target',
     'warning: Observer watches `order-repo` but no feature set stores into it'),

    # ── State machine bugs ───────────────────────────────────────────────────
    ('accept_invalid_transition',
     'Change an Accept statement to transition to a state not defined in the state machine',
     'warning: State `cancelled` is not reachable from current state `pending`'),
    ('where_guard_wrong_field',
     'Change the field name in a `where <field> = value` guard so it references a non-existent event field',
     'warning: Field `status` not found in event schema'),

    # ── File operation bugs ──────────────────────────────────────────────────
    ('read_nonexistent_path',
     'Hardcode a file path string that does not exist in the Read statement',
     'Runtime: FileNotFoundError -- cannot read from non-existent path'),
    ('write_missing_data',
     'Remove the data variable from a Write statement: `Write to the <file>` without specifying what to write',
     'error: Write requires a result (data to write) and an object (destination)'),

    # ── Compute & arithmetic bugs ────────────────────────────────────────────
    ('division_by_zero_risk',
     'Add a Compute statement that divides by a variable that could be zero without a guard',
     'Runtime: Division by zero -- add a `when <divisor> != 0` guard'),
    ('wrong_arithmetic_op',
     'Change a `*` to `++` in an arithmetic Compute (e.g. `Compute the <total> from <price> ++ <qty>`)',
     'Type error: ++ is string concatenation, not multiplication -- use * for arithmetic'),
    ('compute_undefined_var',
     'Reference an undefined variable in a Compute expression',
     'error: Variable `discount` is not defined in this scope'),

    # ── Hallucinated / non-existent action verbs ─────────────────────────────
    # The model invents plausible-sounding actions that don't exist in ARO.
    # These pairs teach it to recognise and fix such hallucinations.
    ('hallucinated_action_tail',
     'Replace a Read or Log statement with a non-existent `Tail` action (e.g. `Tail the <data> from the <file>`)',
     'error: Unknown action verb `Tail` -- did you mean Read or Log?'),
    ('hallucinated_action_scan',
     'Replace a Retrieve or Extract statement with a non-existent `Scan` action (e.g. `Scan the <items> from the <source>`)',
     'error: Unknown action verb `Scan` -- did you mean Retrieve or Extract?'),
    ('hallucinated_action_peek',
     'Replace a Read or Retrieve statement with a non-existent `Peek` action (e.g. `Peek the <value> from the <queue>`)',
     'error: Unknown action verb `Peek` -- did you mean Read or Retrieve?'),
    ('hallucinated_action_update',
     'Replace a Store or Compute statement with a non-existent `Update` action (e.g. `Update the <user> in the <repository>`)',
     'error: Unknown action verb `Update` -- did you mean Store or Compute?'),
    ('hallucinated_action_set',
     'Replace a Compute or Store statement with a non-existent `Set` action (e.g. `Set the <value> to the <target>`)',
     'error: Unknown action verb `Set` -- did you mean Compute or Store?'),
    ('hallucinated_action_query',
     'Replace a Retrieve statement with a non-existent `Query` action (e.g. `Query the <results> from the <database>`)',
     'error: Unknown action verb `Query` -- did you mean Retrieve?'),
    ('hallucinated_action_print',
     'Replace a Log statement with `Print` but capitalised as a new verb `Print the <msg> to the <output>`',
     'error: Unknown action verb `Print` -- did you mean Log? (note: `print` lowercase is a valid Log alias)'),
    ('hallucinated_action_open',
     'Replace a Read or Connect statement with a non-existent `Open` action (e.g. `Open the <connection> to the <server>`)',
     'error: Unknown action verb `Open` -- did you mean Connect or Read?'),
]

print(f'\n--- Task 3: Debugging (target={TARGETS["debugging"]}) ---')
print(f'    Bug types: {len(BUG_TYPES)}')
count = 0

# Build a few-shot reference snippet for debugging prompts
_debug_few_shot = FEW_SHOT_BLOCK + '\n\n' if FEW_SHOT_BLOCK else ''

for ex in (examples * 20)[:TARGETS['debugging'] * 4]:
    if count >= TARGETS['debugging']:
        break
    key = ('debugging', f'{ex["name"]}_{count}')
    if key in done:
        continue
    code = list(ex['aro_files'].values())[0] if ex['aro_files'] else ''
    if len(code) < 100:
        continue

    bug_id, bug_instr, typical_error = random.choice(BUG_TYPES)

    # ── Format 1: Introduce bug + show fix (original format) ─────────────────
    instruction = (
        f'{_debug_few_shot}'
        f'Here is valid ARO code:\n\n```aro\n{code[:2000]}\n```\n\n'
        f'Introduce exactly one bug by doing: {bug_instr}\n\n'
        f'Respond with:\n## Buggy Code\n```aro\n...\n```\n\n'
        f'## Error Message\n(what aro check or the runtime would report)\n\n'
        f'## Fix\n```aro\n...\n```\n\n## Explanation\n(one sentence)'
    )
    output, attempts = generate_with_repair(instruction, max_tokens=1200)
    repair_stats[attempts] += 1

    if output:
        # Save as bug-introduction pair
        save({
            'task_type': 'debugging',
            'instruction': 'Find and fix the bug in this ARO code.',
            'input': code[:2000],
            'output': output,
            'bug_type': bug_id,
            'source': ex['name'],
        })

        # ── Format 2: Error message + buggy code -> fixed code ────────────────
        # Extract the buggy code and error from the model's output for a
        # separate training pair where the user prompt is the error message.
        buggy_match = re.search(r'## Buggy Code\s*```aro\n(.*?)```', output, re.DOTALL)
        fix_match   = re.search(r'## Fix\s*```aro\n(.*?)```', output, re.DOTALL)
        error_match = re.search(r'## Error Message\s*(.*?)(?=\n## |\Z)', output, re.DOTALL)

        if buggy_match and fix_match:
            buggy_code = buggy_match.group(1).strip()
            fixed_code = fix_match.group(1).strip()
            error_msg  = error_match.group(1).strip() if error_match else typical_error

            error_prompt = (
                f'I ran `aro check` on my code and got this error:\n\n'
                f'```\n{error_msg}\n```\n\n'
                f'Here is the code:\n\n```aro\n{buggy_code}\n```\n\n'
                f'Fix the error.'
            )
            save({
                'task_type': 'debugging',
                'instruction': error_prompt,
                'output': f'```aro\n{fixed_code}\n```',
                'bug_type': f'{bug_id}_error_fix',
                'source': f'{ex["name"]}_errfix',
            })

        done.add(key)
        count += 1
        print(f'  [{count}] (attempts={attempts+1}) {ex["name"]} / {bug_id}')

print(f'Done: {count} samples (+ up to {count} error->fix pairs)')

In [ ]:
# Show sample of generated debugging pairs
_dbg_pairs = []
with open(OUTPUT_FILE) as _f:
    for _line in _f:
        if _line.strip():
            _s = json.loads(_line)
            if _s.get('task_type') == 'debugging':
                _dbg_pairs.append(_s)
_show_sample(_dbg_pairs, label='NB10 debugging')

## Task 4 — Syntax Q&A

In [ ]:
print(f'\n--- Task 4: Syntax Q&A (target={TARGETS["syntax_qa"]}) ---')
all_seeds = [s for p in kb['proposals'] for s in p['qa_seeds']]
count = 0
for seed in random.sample(all_seeds * 3, min(TARGETS['syntax_qa'] * 3, len(all_seeds) * 3)):
    if count >= TARGETS['syntax_qa']:
        break
    heading = seed.get('heading', '')
    key = ('syntax_qa', heading)
    if key in done or len(seed['body']) < 80:
        continue
    code_ctx = f'\n\nExample:\n```aro\n{seed["code_examples"][0][:400]}\n```' if seed.get('code_examples') else ''
    messages = [
        {'role': 'system', 'content': ARO_SYSTEM_PROMPT},
        {'role': 'user',   'content': f'Based on this ARO documentation section "{heading}":\n\n{seed["body"][:1200]}{code_ctx}\n\nGenerate one question-and-answer pair.\n\n## Question\n...\n\n## Answer\n...'},
    ]
    output = chat(messages, max_tokens=500)
    q_m = re.search(r'## Question\n(.+?)(?=\n##|\Z)', output, re.DOTALL)
    save({'task_type': 'syntax_qa', 'instruction': q_m.group(1).strip() if q_m else heading, 'output': output, 'proposal': seed['proposal'], 'heading': heading})
    done.add(key)
    count += 1
    print(f'  [{count}] {heading[:70]}')

print(f'Done: {count} samples')

In [ ]:
# Show sample of generated syntax_qa pairs
_qa_pairs = []
with open(OUTPUT_FILE) as _f:
    for _line in _f:
        if _line.strip():
            _s = json.loads(_line)
            if _s.get('task_type') == 'syntax_qa':
                _qa_pairs.append(_s)
_show_sample(_qa_pairs, label='NB10 syntax_qa')

## Task 5 — Translation

In [ ]:
if False:  # DISABLED — 2.5% success rate
    STYLES = ['Python Flask pseudocode', 'Express.js route handlers', 'plain English description', 'SQL query', 'Apache Spark function']

    print(f'\n--- Task 5: Translation (target={TARGETS.get("translation", 0)}) ---')
    count = 0
    for domain in (DOMAINS * 5)[:TARGETS.get('translation', 0) * 3]:
        if count >= TARGETS.get('translation', 0):
            break
        key = ('translation', domain)
        if key in done:
            continue
        style = random.choice(STYLES)
        instruction = (
            f'First write a brief {style} for: {domain}.\n'
            f'Then translate it into a complete ARO application.\n\n'
            f'## Original ({style})\n```\n...\n```\n\n## ARO Translation\n```aro\n...\n```'
        )
        output, attempts = generate_with_repair(instruction)
        repair_stats[attempts] += 1
        if output:
            save({'task_type': 'translation', 'instruction': f'Translate this {style} into an ARO application.', 'output': output, 'domain': domain})
            done.add(key)
            count += 1
            print(f'  [{count}] (attempts={attempts+1}) {domain[:55]} ({style})')

    print(f'Done: {count} samples')

print('Task 5 (Translation): DISABLED — 2.5% success rate')

## Task 6 — Architecture

In [ ]:
# ── Task 6: Architecture — multi-feature-set applications ─────────────────────
# These teach the model to build complete applications with multiple feature sets,
# event handlers, and state machines — all self-contained and runnable.

SCENARIOS = [
    ('order fulfillment with state transitions (draft → placed → shipped → delivered)',
     'Create orders, Accept state transitions, Emit events on state change, Log each transition'),
    ('user registration with email notification',
     'Extract user data from request, Validate email, Store user, Emit UserCreated event, Send welcome email in handler'),
    ('inventory tracker that alerts on low stock',
     'Store products with quantities, Compute stock level, Compare against threshold, Emit LowStock event when below 10'),
    ('task manager with priority sorting',
     'Create tasks with priority field, Store into repository, Retrieve all tasks, Sort by priority, Log sorted list'),
    ('event-driven data pipeline: filter, transform, aggregate',
     'Create data list, Filter where value > threshold, Transform with uppercase, Compute count and sum, Log results'),
    ('shopping cart with item add, remove, and total calculation',
     'Create cart items, Store into cart-repository, Retrieve all items, Compute total from prices, Delete item by id'),
    ('notification system with multiple channels (console, log)',
     'Emit Notification event with message and channel, Handle console channel, Handle log channel, Log delivery'),
    ('file metadata indexer that scans and stores file info',
     'Create file list, For each file Compute hash, Create metadata object, Store into index-repository, Log indexed count'),
    ('multi-step validation pipeline',
     'Extract data, Validate email format, Validate age range, Compare password fields, Return OK or throw on failure'),
    ('rate counter that tracks events per category',
     'Extract category from event, Retrieve counter from repository, Compute incremented count, Update counter, Log totals'),
]

print(f'\n--- Task 6: Architecture (target={TARGETS.get("architecture", 0)}) ---')
count = 0
for scenario_desc, actions_hint in (SCENARIOS * 5)[:TARGETS.get('architecture', 0) * 3]:
    if count >= TARGETS.get('architecture', 0):
        break
    key = ('architecture', scenario_desc)
    if key in done:
        continue

    instruction = (
        f'Write a complete, self-contained ARO application that implements: {scenario_desc}.\n\n'
        f'The application must:\n'
        f'- Have exactly one Application-Start feature set\n'
        f'- Include at least 2 additional feature sets (event handlers or API endpoints)\n'
        f'- Use these actions: {actions_hint}\n'
        f'- Be fully runnable with `aro run` (no external files, URLs, or services)\n'
        f'- Use Log statements so the output is visible\n'
        f'- End every feature set with Return\n\n'
        f'Output only the ARO source code in ```aro fences.'
    )
    output, attempts = generate_with_repair(instruction, max_tokens=1500)
    repair_stats[attempts] += 1
    if output:
        save({
            'task_type': 'architecture',
            'instruction': f'Write a complete ARO application that implements: {scenario_desc}.',
            'output': output,
            'scenario': scenario_desc,
        })
        done.add(key)
        count += 1
        print(f'  [{count}] (attempts={attempts+1}) {scenario_desc[:65]}')

print(f'Done: {count} architecture samples')


## Task 7 — Fill-in-the-Middle (FIM)

In [ ]:
print(f'\n--- Task 7: Fill-in-the-Middle (target={TARGETS["fim"]}) ---')
count = 0

# Strategy: mask the ENTIRE body of one feature set, keeping its header and
# all other feature sets as context.  This teaches holistic understanding of
# feature set structure — much harder and more useful than masking 1-3 lines.
# Fallback to line-masking for files with only one tiny feature set.
_fs_body_re = re.compile(
    r'(\([^)]+:\s*[^)]+\)\s*\{)(.*?)(\})',
    re.DOTALL,
)

for ex in (examples * 20)[:TARGETS['fim'] * 5]:
    if count >= TARGETS['fim']:
        break

    # Concatenate all .aro files in this example
    code = '\n\n'.join(ex['aro_files'].values())

    # Find feature sets whose body is non-trivial (>20 non-whitespace chars)
    fs_matches = [
        (m.group(1), m.group(2), m.group(3), m.start(), m.end())
        for m in _fs_body_re.finditer(code)
        if len(m.group(2).strip()) > 20
    ]

    if fs_matches:
        # Whole-feature-set masking: model must predict the entire body
        hdr, body, close, start, end = random.choice(fs_matches)
        prefix      = code[:start] + hdr
        middle      = body                     # full body — the fill target
        suffix      = close + code[end:]
        instruction = 'Complete the missing ARO feature set body.'
    else:
        # Fallback: mask 1–3 lines (for tiny single-feature-set files)
        lines = [l for l in code.split('\n') if l.strip() and not l.strip().startswith('(*')]
        if len(lines) < 6:
            continue
        split  = random.randint(len(lines) // 4, 3 * len(lines) // 4)
        mask   = random.randint(1, min(3, len(lines) - split - 1))
        prefix = '\n'.join(lines[:split])
        middle = '\n'.join(lines[split:split + mask])
        suffix = '\n'.join(lines[split + mask:])
        instruction = 'Complete the missing ARO statement(s).'

    if not middle.strip():
        continue

    save({
        'task_type':   'fim',
        'instruction': instruction,
        'prefix':      prefix,
        'suffix':      suffix,
        'middle':      middle,
        'source':      ex['name'],
    })
    count += 1
    if count % 50 == 0:
        print(f'  [{count}] FIM samples')

print(f'Done: {count} samples')

In [ ]:
# Show sample of generated fim pairs
_fim_pairs = []
with open(OUTPUT_FILE) as _f:
    for _line in _f:
        if _line.strip():
            _s = json.loads(_line)
            if _s.get('task_type') == 'fim':
                _fim_pairs.append(_s)
_show_sample(_fim_pairs, label='NB10 fim')

## Task 8 — Correction ("that action doesn't exist")

Generate training pairs where the user asks about a non-existent action
and the model explains which real action(s) to use instead.
This directly combats hallucinated action verbs like Tail, Scan, Update, etc.

In [ ]:
# ── Task 8: Correction — teach the model that certain actions do NOT exist ──
# Each pair: user asks about a fake action → assistant explains what to use instead.

FAKE_ACTIONS = [
    # (fake_verb, user_question, correct_answer)
    ('Tail',
     'Can I tail a file in ARO?',
     'ARO does not have a `Tail` action. To watch a file for changes, use the '
     'file-monitor pattern:\n\n```aro\n(Application-Start: File Watcher) {\n'
     '    Start the <file-monitor> with ".".\n'
     '    Keepalive the <application> for the <events>.\n'
     '    Return an <OK: status> for the <startup>.\n}\n\n'
     '(Handle File Modified: File Event Handler) {\n'
     '    Extract the <path> from the <event: path>.\n'
     '    Log <path> to the <console>.\n'
     '    Return an <OK: status> for the <event>.\n}\n```\n\n'
     'To simply read a file, use `Read the <content> from the <file: "path">.`'),

    ('Scan',
     'How do I scan items from a data source in ARO?',
     'There is no `Scan` action in ARO. Use `Retrieve` to fetch data from a repository '
     'or `Extract` to get fields from an object:\n\n```aro\nRetrieve the <items> from '
     'the <item-repository>.\nExtract the <name> from the <item: name>.\n```'),

    ('Update',
     'How do I update a record in ARO?',
     'ARO does not have an `Update` action. Variables are immutable — create a new value '
     'and store it:\n\n```aro\nRetrieve the <user> from the <user-repository> where '
     'id = <id>.\nCreate the <updated-user> with <user> and <new-data>.\n'
     'Store the <updated-user> into the <user-repository>.\n```'),

    ('Set',
     'Can I set a variable value in ARO?',
     'There is no `Set` action. ARO variables are immutable and bound once via an action. '
     'Use `Compute` for transformations or `Extract` to bind a value:\n\n```aro\n'
     'Compute the <total> from <price> * <quantity>.\n'
     'Extract the <name> from the <user: name>.\n```'),

    ('Query',
     'How do I query a database in ARO?',
     'ARO does not have a `Query` action. Use `Retrieve` with a `where` clause to fetch '
     'filtered data:\n\n```aro\nRetrieve the <active-users> from the <user-repository> '
     'where status = "active".\n```'),

    ('Peek',
     'Can I peek at a value without consuming it in ARO?',
     'There is no `Peek` action in ARO. Use `Read` to read file contents or `Retrieve` to '
     'fetch from a repository. Both are non-destructive:\n\n```aro\n'
     'Read the <content> from the <file: "data.json">.\n'
     'Retrieve the <item> from the <item-repository> where id = <id>.\n```'),

    ('Open',
     'How do I open a connection in ARO?',
     'ARO does not have an `Open` action. Use `Connect` for TCP connections or `Start` '
     'for services:\n\n```aro\nConnect the <connection> to the <server: "localhost:8080">.\n'
     'Start the <http-server> with <contract>.\n```'),

    ('Print',
     'How do I print output in ARO?',
     'Use the `Log` action to print to the console. Note: `print` is a valid alias for Log, '
     'but the canonical verb is `Log`:\n\n```aro\n'
     'Log "Hello, world!" to the <console>.\n'
     'Log <user-name> to the <console>.\n```'),

    ('Get',
     'Can I use Get to fetch data in ARO?',
     '`Get` is an alias for `Extract`, so it works — but the canonical verbs are '
     '`Retrieve` (from repositories) and `Extract` (from objects):\n\n```aro\n'
     'Retrieve the <users> from the <user-repository>.\n'
     'Extract the <email> from the <user: email>.\n```'),

    ('Insert',
     'How do I insert a record into a repository in ARO?',
     'There is no `Insert` action. Use `Store` to persist data into a repository:\n\n'
     '```aro\nCreate the <user> with <data>.\n'
     'Store the <user> into the <user-repository>.\n```'),

    ('Remove',
     'Is there a Remove action in ARO?',
     '`Remove` is a valid alias for `Delete`. You can use either:\n\n```aro\n'
     'Delete the <item> from the <item-repository> where id = <id>.\n```'),

    ('Watch',
     'How do I watch a directory for changes in ARO?',
     'There is no `Watch` action. Use the file-monitor service pattern:\n\n```aro\n'
     'Start the <file-monitor> with "./watched-dir".\n'
     'Keepalive the <application> for the <events>.\n```\n\n'
     'Then handle events with `File Event Handler` feature sets.'),

    ('Subscribe',
     'Can I subscribe to events in ARO?',
     '`Subscribe` is a valid alias for `Stream`. But for most event handling, you '
     'don\'t need to subscribe explicitly — just name your feature set as a Handler:\n\n'
     '```aro\n(Handle Order: OrderCreated Handler) {\n'
     '    Extract the <order> from the <event: order>.\n'
     '    Log <order> to the <console>.\n'
     '    Return an <OK: status> for the <event>.\n}\n```'),
]

print(f'\n--- Task 8: Correction (target={TARGETS["correction"]}) ---')
print(f'    Fake action templates: {len(FAKE_ACTIONS)}')
count = 0

# Build few-shot context
_corr_few_shot = FEW_SHOT_BLOCK + '\n\n' if FEW_SHOT_BLOCK else ''

# Deterministic pairs from templates (no LLM call needed)
for fake_verb, question, answer in FAKE_ACTIONS:
    if count >= TARGETS['correction']:
        break
    key = ('correction', f'template_{fake_verb}')
    if key in done:
        continue
    save({
        'task_type': 'correction',
        'instruction': question,
        'output': answer,
        'source': f'template_{fake_verb}',
    })
    done.add(key)
    count += 1
    print(f'  [{count}] template: {fake_verb}')

# Generate additional correction pairs with LLM for variety
_extra_fake_verbs = [
    'Tail', 'Scan', 'Update', 'Set', 'Query', 'Peek', 'Open', 'Insert',
    'Watch', 'Push', 'Pull', 'Grab', 'Put', 'Dump', 'Load', 'Pipe',
    'Mount', 'Attach', 'Link', 'Bind', 'Assign', 'Define', 'Declare',
]

for fake_verb in _extra_fake_verbs:
    if count >= TARGETS['correction']:
        break
    key = ('correction', f'generated_{fake_verb}')
    if key in done:
        continue

    instruction = (
        f'{_corr_few_shot}'
        f'A user wrote ARO code using a `{fake_verb}` action, but `{fake_verb}` does not '
        f'exist in ARO. The valid actions are listed in the system prompt above.\n\n'
        f'Write a short, helpful response that:\n'
        f'1. Explains that `{fake_verb}` is not a valid ARO action\n'
        f'2. Suggests which real ARO action(s) to use instead\n'
        f'3. Shows a corrected code example in ```aro fences\n\n'
        f'Keep it concise (under 150 words).'
    )
    output, attempts = generate_with_repair(instruction, max_tokens=500,
                                            skip_aro_check=True)
    repair_stats[attempts] += 1
    if output:
        save({
            'task_type': 'correction',
            'instruction': f'I tried to use `{fake_verb}` in my ARO code but it doesn\'t work. '
                           f'What should I use instead?',
            'output': output,
            'source': f'generated_{fake_verb}',
        })
        done.add(key)
        count += 1
        print(f'  [{count}] generated: {fake_verb} (attempts={attempts+1})')

print(f'Done: {count} correction samples')


## Task 9 — Full Application (plan → complete multi-file app)

For each ARO example/application that has a `plan.md`, generate 9 alternative
phrasings of the plan and pair all 10 with the complete application code.
This teaches the model to read a detailed implementation plan and produce
a complete, multi-file ARO application.

In [ ]:
# ── Task 9: Full Application — plan.md → all files ────────────────────────────
# Scan Examples/ and ARO-Application/ for plan.md files.
# For each: build the "golden output" (all .aro + openapi.yaml with filenames),
# save the original plan as one pair, then generate 9 rephrasings and save those.

import glob as _glob

_APP_DIRS = [
    EXAMPLES_DIR,           # Examples/ in ARO-Train
    ARO_APPLICATION_ROOT,   # ../ARO-Application
]

def _collect_plan_apps():
    """Find all directories with a plan.md and at least one .aro file."""
    apps = []
    for root in _APP_DIRS:
        if not root.exists():
            continue
        for plan_path in sorted(root.rglob('plan.md')):
            app_dir = plan_path.parent
            aro_files = sorted(app_dir.glob('*.aro'))
            if not aro_files:
                continue
            apps.append({
                'name': app_dir.name,
                'dir': app_dir,
                'plan': plan_path.read_text().strip(),
                'aro_files': {f.name: f.read_text() for f in aro_files},
                'openapi': (app_dir / 'openapi.yaml').read_text()
                           if (app_dir / 'openapi.yaml').exists() else None,
                'store_files': {f.name: f.read_text()
                                for f in sorted(app_dir.glob('*.store'))},
            })
    return apps


def _build_golden_output(app):
    """Format all application files as the expected model output."""
    parts = []
    # openapi.yaml first (contract-first)
    if app['openapi']:
        parts.append(f'**openapi.yaml**\n```yaml\n{app["openapi"]}\n```')
    # .store files
    for name, content in app['store_files'].items():
        parts.append(f'**{name}**\n```yaml\n{content}\n```')
    # .aro files (main.aro first)
    aro_sorted = sorted(app['aro_files'].keys(),
                        key=lambda n: (0 if n == 'main.aro' else 1, n))
    for name in aro_sorted:
        parts.append(f'**{name}**\n```aro\n{app["aro_files"][name]}\n```')
    return '\n\n'.join(parts)


# ── Collect applications ──────────────────────────────────────────────────────
_plan_apps = _collect_plan_apps()
print(f'\n--- Task 9: Full Application (plan → complete app) ---')
print(f'    Found {len(_plan_apps)} applications with plan.md')
for a in _plan_apps:
    n_files = len(a['aro_files']) + (1 if a['openapi'] else 0) + len(a['store_files'])
    print(f'    {a["name"]:30s} {n_files} files')

count = 0
REPHRASINGS_PER_APP = 9

for app in _plan_apps:
    golden = _build_golden_output(app)
    plan = app['plan']

    # Skip if golden output is too long for the context window
    if len(plan) + len(golden) > 14000:
        print(f'  [skip] {app["name"]} — too large ({len(golden)} chars)')
        continue

    # ── Save original plan as first pair ──────────────────────────────────────
    key = ('full_application', f'{app["name"]}_original')
    if key not in done:
        save({
            'task_type': 'full_application',
            'instruction': plan,
            'output': golden,
            'source': f'plan:{app["name"]}',
        })
        done.add(key)
        count += 1
        print(f'  [{count}] {app["name"]} (original plan)')

    # ── Generate alternative phrasings ────────────────────────────────────────
    for variant_idx in range(REPHRASINGS_PER_APP):
        key = ('full_application', f'{app["name"]}_v{variant_idx}')
        if key in done:
            continue

        rephrase_prompt = (
            f'Rephrase the following ARO application implementation plan. '
            f'Keep all technical details (endpoints, event names, file names, '
            f'data models) but change the wording, sentence structure, and '
            f'ordering. Make it sound like a different person wrote it. '
            f'Variant {variant_idx + 1} of {REPHRASINGS_PER_APP} — '
            f'each variant should be distinctly different.\n\n'
            f'Original plan:\n\n{plan}'
        )
        rephrased = chat(
            [{'role': 'user', 'content': rephrase_prompt}],
            max_tokens=len(plan) // 2 + 500,
        ).strip()

        if len(rephrased) < 50:
            print(f'  [skip] {app["name"]} v{variant_idx} — rephrasing too short')
            continue

        # Strip any markdown wrapper the model might add
        if rephrased.startswith('```'):
            rephrased = re.sub(r'^```\w*\n', '', rephrased)
            rephrased = re.sub(r'\n```$', '', rephrased)

        save({
            'task_type': 'full_application',
            'instruction': rephrased,
            'output': golden,
            'source': f'plan:{app["name"]}_v{variant_idx}',
        })
        done.add(key)
        count += 1
        print(f'  [{count}] {app["name"]} v{variant_idx}')

print(f'Done: {count} full_application samples '
      f'({len(_plan_apps)} apps × up to {1 + REPHRASINGS_PER_APP} variants)')


## Summary

In [ ]:
all_samples = []
with open(OUTPUT_FILE) as f:
    for line in f:
        if line.strip():
            all_samples.append(json.loads(line))

counts = Counter(s['task_type'] for s in all_samples)
print(f'\nTotal samples: {len(all_samples)}')
for task, n in sorted(counts.items(), key=lambda x: -x[1]):
    print(f'  {task:25s}: {n:4d}  (target: {TARGETS.get(task, "?")})')

print(f'\nSelf-repair stats (0 = passed on first try):')
for attempts, n in sorted(repair_stats.items()):
    label = 'discarded' if attempts == MAX_REPAIR_ATTEMPTS else f'{attempts+1} attempt(s)'
    print(f'  {label}: {n}')

# Save repair rejects for DPO training (broken code the model produced during repair loops)
_repair_rejects_file = DATA_OUT / 'repair_rejects.jsonl'
if _repair_rejects:
    with open(_repair_rejects_file, 'w') as f:
        for r in _repair_rejects:
            f.write(json.dumps(r) + '\n')
    print(f'\nRepair rejects: {len(_repair_rejects)} saved to {_repair_rejects_file}')
    print('  These will be used as DPO negatives in NB17.')
else:
    print('\nNo repair rejects to save.')


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222',
    'axes.labelcolor': '#222222',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.titlecolor': '#111111',
    'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
})
from pathlib import Path
import numpy as np
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '10_synthetic_data_generation.png'

_tasks  = sorted(TARGETS.keys())
_target = [TARGETS[t] for t in _tasks]
_actual = [counts.get(t, 0) for t in _tasks]

x = np.arange(len(_tasks))
w = 0.38

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w/2, _target, w, label='Target', color='#bdc3c7', edgecolor='white')
b2 = ax.bar(x + w/2, _actual, w, label='Generated',
            color=['#2ecc71' if a >= t else '#e74c3c'
                   for a, t in zip(_actual, _target)],
            edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels([t.replace('_', '\n') for t in _tasks], fontsize=9)
ax.set_ylabel('Samples')
ax.set_title(
    f'Synthetic Data Generation — {len(all_samples):,} total samples across {len(TARGETS)} task types',
    fontsize=13, fontweight='bold'
)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
for bar in b2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 1, str(int(h)),
            ha='center', va='bottom', fontsize=8, color='#333')
fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')

# ── CSV export: per-sample results for downstream analysis ────────────────────
import csv

csv_path = _run_dir / '10_synthetic_data_generation.csv'
with open(csv_path, 'w', newline='') as csvf:
    writer = csv.DictWriter(csvf, fieldnames=[
        'prompt', 'task_type', 'aro_check_pass', 'reject_reason', 'attempt_number',
    ])
    writer.writeheader()
    for s in all_samples:
        writer.writerow({
            'prompt':          s.get('instruction', s.get('domain', ''))[:500],
            'task_type':       s.get('task_type', ''),
            'aro_check_pass':  True,       # all saved samples passed
            'reject_reason':   '',
            'attempt_number':  s.get('attempts', 0),
        })
    # Also write failed prompts
    for fp in _failed_prompts:
        writer.writerow({
            'prompt':          fp.get('instruction', '')[:500],
            'task_type':       'code_generation',
            'aro_check_pass':  False,
            'reject_reason':   fp.get('reason', 'unknown'),
            'attempt_number':  MAX_REPAIR_ATTEMPTS,
        })
print(f'CSV export: {csv_path}  ({len(all_samples)} passed + {len(_failed_prompts)} failed)')

if PILOT_MODE:
    _pass_rate = len(all_samples) / max(1, len(all_samples) + len(_failed_prompts)) * 100
    print(f'\n{"="*60}')
    print(f'PILOT MODE REPORT')
    print(f'  Samples generated : {len(all_samples)}')
    print(f'  Samples failed    : {len(_failed_prompts)}')
    print(f'  Pass rate         : {_pass_rate:.1f}%')
    print(f'  CSV               : {csv_path}')
    print(f'{"="*60}')
    print('Set PILOT_MODE = False and rerun for full generation.')


In [ ]:
# ── Samples per category ─────────────────────────────────────────────────────
import json, random
from collections import defaultdict

_pairs = []
if OUTPUT_FILE.exists():
    with open(OUTPUT_FILE) as f:
        for line in f:
            if line.strip():
                _pairs.append(json.loads(line))

_by_type = defaultdict(list)
for p in _pairs:
    _by_type[p.get('task_type', 'unknown')].append(p)

_CATEGORY_ORDER = [
    'code_generation',
    'code_explanation',
    'debugging',
    'syntax_qa',
    'translation',
    'architecture',
    'fim',
]
SAMPLES_PER_CAT = 2

for task_type in _CATEGORY_ORDER:
    pool = _by_type.get(task_type, [])
    if not pool:
        continue
    print(f'\n{"─"*72}')
    print(f'  {task_type}  ({len(pool)} pairs)')
    print('─'*72)
    for s in random.sample(pool, min(SAMPLES_PER_CAT, len(pool))):
        print(f'Q: {s.get("instruction", next((m["content"] for m in s.get("messages",[]) if m.get("role")=="user"), ""))[:220]}')
        out = s.get('output', '')
        print(f'A: {out[:320]}{"..." if len(out) > 320 else ""}')
        print()

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222',
    'axes.labelcolor': '#222222',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.titlecolor': '#111111',
    'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
})
# ── Final status: RL fine-tune loss curves ────────────────────────────────────
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '10_rl_loss.png'

if not _rl_losses:
    print('No RL fine-tune rounds completed — skipping loss graph.')
    print(f'  Rounds run  : {_rl_round}')
    print(f'  RL adapter  : {_rl_adapter}')
else:
    _palette = plt.cm.Blues(
        [0.4 + 0.6 * i / max(1, len(_rl_losses) - 1) for i in range(len(_rl_losses))]
    )
    fig, ax = plt.subplots(figsize=(10, 5))

    for idx, (rnd, data) in enumerate(sorted(_rl_losses.items())):
        color = _palette[idx]
        if data['train_losses']:
            ax.plot(data['train_iters'], data['train_losses'],
                    '-o', ms=3, lw=1.5, color=color, label=f'Round {rnd} train')
        if data['val_losses']:
            ax.plot(data['val_iters'], data['val_losses'],
                    's--', ms=6, lw=1.5, color=color, alpha=0.7, label=f'Round {rnd} val')

    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')
    ax.set_title(
        f'RL Fine-Tune Loss — {len(_rl_losses)} round(s)  ·  {LORA_LAYERS} LoRA layers',
        fontsize=13, fontweight='bold', pad=12,
    )
    ax.legend(fontsize=9, ncol=2, framealpha=1.0, facecolor='white', edgecolor='#ccc')
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(_out)
    plt.close(fig)
    print(f'Saved: {_out}')
    print(f'  Rounds completed : {len(_rl_losses)}')
    print(f'  Final adapter    : {_rl_adapter}')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import json, re
from pathlib import Path
from collections import Counter
from datetime import date as _date

plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222', 'axes.labelcolor': '#222222',
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'axes.titlecolor': '#111111', 'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white', 'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa', 'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa', 'savefig.bbox': 'tight', 'savefig.dpi': 150,
})

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '10_action_coverage.png'

# ── Extract verb usage from accepted and rejected samples ────────────────────
def _extract_verbs(text):
    return set(
        m.group(1).lower()
        for m in re.finditer(r'^\s*([A-Z][a-z]+)\s+(?:the|a|an)?\s*<', text, re.MULTILINE)
    )

_accepted_verbs = Counter()
_rejected_verbs = Counter()

# Accepted
with open(OUTPUT_FILE) as _f:
    for _line in _f:
        if _line.strip():
            _s = json.loads(_line)
            for _v in _extract_verbs(_s.get('output', '')):
                _accepted_verbs[_v] += 1

# Rejected
_reject_file = DATA_OUT / 'repair_rejects.jsonl'
if _reject_file.exists():
    with open(_reject_file) as _f:
        for _line in _f:
            if _line.strip():
                _r = json.loads(_line)
                for _v in _extract_verbs(_r.get('output', '')):
                    _rejected_verbs[_v] += 1

# ── Build chart data: sort by total usage descending ────────────────────────
_all_verbs = sorted(
    set(_accepted_verbs) | set(_rejected_verbs),
    key=lambda v: -(_accepted_verbs[v] + _rejected_verbs[v])
)

# Show top 30 verbs (chart gets unreadable with 70+)
_top = _all_verbs[:30]
_acc = [_accepted_verbs[v] for v in _top]
_rej = [_rejected_verbs[v] for v in _top]

fig, ax = plt.subplots(figsize=(16, 6))
x = np.arange(len(_top))
w = 0.7

ax.bar(x, _acc, w, label='Accepted', color='#2ecc71', edgecolor='white')
ax.bar(x, _rej, w, bottom=_acc, label='Rejected', color='#e74c3c', edgecolor='white', alpha=0.7)

# Add accept rate labels on top
for i, (a, r) in enumerate(zip(_acc, _rej)):
    total = a + r
    if total > 0:
        rate = 100 * a / total
        ax.text(i, total + max(_acc) * 0.01, f'{rate:.0f}%',
                ha='center', va='bottom', fontsize=7, fontweight='bold',
                color='#27ae60' if rate >= 60 else '#e74c3c')

ax.set_xticks(list(x))
ax.set_xticklabels([v.capitalize() for v in _top], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Samples containing verb')
ax.set_title(
    f'Action Verb Coverage — {len(_all_verbs)} verbs across '
    f'{sum(_accepted_verbs.values()):,} accepted + {sum(_rejected_verbs.values()):,} rejected uses',
    fontsize=12, fontweight='bold',
)
ax.legend(fontsize=9, loc='upper right')
ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')

# Print coverage summary
_zero_accepted = [v for v in _all_verbs if _accepted_verbs[v] == 0]
_low_rate = [v for v in _all_verbs if _accepted_verbs[v] + _rejected_verbs[v] > 5
             and _accepted_verbs[v] / max(1, _accepted_verbs[v] + _rejected_verbs[v]) < 0.4]
print(f'\nVerbs with 0 accepted: {len(_zero_accepted)}  {_zero_accepted}')
print(f'Verbs with <40% accept rate (>5 samples): {len(_low_rate)}  {_low_rate}')
print(f'Verbs not seen at all: {len(set(_verb_list) - set(_all_verbs))}  '
      f'{sorted(set(v.lower() for v in _verb_list) - set(_all_verbs))}')
